In [1]:
from transformers import pipeline
from datasets import load_dataset
from huggingface_hub import list_repo_files

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [3]:
import numpy as np

In [4]:
files = list_repo_files("HuggingFaceM4/WebSight", repo_type="dataset")
train_files = [f"https://huggingface.co/datasets/HuggingFaceM4/WebSight/resolve/main/{f}"
               for f in files if "data/train" in f and f.endswith(".parquet")]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
dataset = load_dataset("parquet", data_files={"train": train_files[:2]})

data/train-00000-of-00071-eb722b04b83e13(…):   0%|          | 0.00/438M [00:00<?, ?B/s]

data/train-00001-of-00071-df5cc75986b4e6(…):   0%|          | 0.00/443M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
dataset_xs = dataset["train"].shard(num_shards=100, index=0)

In [22]:
from transformers import ViTImageProcessor, ViTModel

In [8]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224', return_tensors="pt")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [18]:
# drop alpha channel
def preprocess(example):
    arr = processor(example["image"].convert("RGB"), return_tensors="np").pixel_values.squeeze()
    # print(type(arr))
    example["pixels"] = arr
    return example

dataset_xs_p = dataset_xs.map(preprocess)


Map:   0%|          | 0/232 [00:00<?, ? examples/s]

In [23]:
model = ViTModel.from_pretrained("google/vit-base-patch16-224")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
ex_img = torch.Tensor(dataset_xs_p["pixels"][5]).unsqueeze(0)

In [43]:
op = model(ex_img)

In [44]:
op.last_hidden_state.shape

torch.Size([1, 197, 768])